# Evaluator — TTS Listen Notebook

Use the evaluator's built-in text-to-speech (TTS) layer to synthesize a few audio
samples from text and **listen to them inline**.

This is handy for sanity-checking the synthetic audio that feeds the
`text question → TTS audio → ASR → retrieval` pipeline before running a full evaluation.

We use the `AudioSynthesizer` + `AudioSynthesisConfig` API directly, with the
**MMS** provider (Meta MMS, runs locally via Hugging Face `transformers` — no separate
binary to install).

## 1. Install

In [ ]:
# Install the evaluator package from repo root
%pip install -q -e '..'

# MMS TTS needs transformers + torch; soundfile to write WAVs
%pip install -q transformers torch soundfile

## 2. Configure the synthesizer

`AudioSynthesisConfig` drives provider, voice/language, sample rate, and caching.
Switch `provider` to `"piper"` or `"xtts_v2"` if those are installed; MMS is the
zero-extra-setup default here.

In [ ]:
from pathlib import Path

from evaluator.config.audio_synthesis import AudioSynthesisConfig
from evaluator.pipeline.audio.synthesis import AudioSynthesizer

OUT_DIR = Path("tts_samples")
OUT_DIR.mkdir(exist_ok=True)

config = AudioSynthesisConfig(
    enabled=True,
    provider="mms",          # "mms" | "piper" | "xtts_v2"
    language="en",           # MMS picks facebook/mms-tts-eng from this
    sample_rate=16000,        # matches typical ASR front-ends
    output_dir=str(OUT_DIR),  # cache auto-derived under here
)

synth = AudioSynthesizer(config)
print("TTS provider ready:", config.provider)

## 3. Synthesize a few samples

We feed a handful of short texts (the kind of queries a retrieval benchmark would
contain) and write each one to a WAV file.

In [ ]:
samples = [
    "Does aspirin reduce the risk of heart attack?",
    "What are the side effects of metformin?",
    "How does sleep deprivation affect memory consolidation?",
]

audios = []
for i, text in enumerate(samples):
    wav_path = OUT_DIR / f"sample_{i}.wav"
    audio = synth.synthesize(text, output_path=str(wav_path))
    audios.append(audio)
    dur = len(audio) / config.sample_rate
    print(f"[{i}] {dur:5.2f}s  ->  {wav_path}   |  {text}")

## 4. Listen to them

Render an inline audio player per sample. Click play to hear the synthesized speech.

In [ ]:
from IPython.display import Audio, display

for i, (text, audio) in enumerate(zip(samples, audios)):
    print(f"[{i}] {text}")
    display(Audio(audio, rate=config.sample_rate))

## 5. (Optional) Try another language or voice

MMS supports many languages via the `language` code (e.g. `pl`, `de`, `fr`, `es`).
You can also point `voice` straight at an HF model id
(e.g. `facebook/mms-tts-pol`).

In [ ]:
pl_config = AudioSynthesisConfig(
    enabled=True,
    provider="mms",
    language="pl",
    sample_rate=16000,
    output_dir=str(OUT_DIR),
)
pl_synth = AudioSynthesizer(pl_config)

pl_text = "Czy aspiryna zmniejsza ryzyko zawału serca?"
pl_audio = pl_synth.synthesize(pl_text, output_path=str(OUT_DIR / "sample_pl.wav"))

from IPython.display import Audio
print(pl_text)
Audio(pl_audio, rate=pl_config.sample_rate)